In [1]:
import sys
import os
import time
import pickle
import cv2
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
import kornia.geometry.subpix.dsnt as dsnt
from kornia.utils.grid import create_meshgrid
import math
from einops.einops import rearrange
from typing import Sequence
import importlib
from utils.misc import setup_gpus, print_params_summary
from default_config import get_cfg_defaults
from src.rcrm_v1 import RCRM_v1
from src.loss import RCRM_Loss
from src.utils.supervision import compute_supervision_coarse, compute_supervision_fine
from utils.metrics import (
    compute_symmetrical_epipolar_errors,
    compute_pose_errors,
    aggregate_metrics,
)
from datasets.rcrm_dataset import RCRM_Dataset

from src.utils.misc import create_grid
from src.utils.geometry import warp_kpts
from utils.misc import print_params_summary

In [2]:
config = get_cfg_defaults()
config.DEVICE.GPU_IDX = "7,"

config.DEVICE.ENABLE_DDP = False
# setup exact gpus available and set CUDA_VISIBLE_DEVICES variable
n_gpu_available = setup_gpus(config.DEVICE.GPU_IDX) if config.DEVICE.ENABLE_GPU else 0
config.TRAINER.WORLD_SIZE = n_gpu_available * config.DEVICE.NUM_NODES
config.TRAINER.TRUE_BATCH_SIZE = config.TRAINER.WORLD_SIZE * config.LOADER.BATCH_SIZE
config.TRAINER.SCALING = config.TRAINER.TRUE_BATCH_SIZE / config.TRAINER.CANONICAL_BS
config.TRAINER.TRUE_LR = config.TRAINER.CANONICAL_LR * config.TRAINER.SCALING
config.TRAINER.WARMUP_STEP = math.floor(
    config.TRAINER.WARMUP_STEP / config.TRAINER.SCALING
)
device = torch.device("cuda:0")

if config["MODEL"]["VERSION"] == "v1":
    _RCRM = RCRM_v1

rcrm = _RCRM(config=config["MODEL"])
print_params_summary(rcrm)
rcrm.to(device)
loss = RCRM_Loss(config.LOSS)
loss.to(device)


# 保存数据
def save_data(datas, save_path):
    # 转换数据
    processed_datas = []
    for data in datas:
        processed_data = {}
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                processed_data[k] = v.cpu().numpy()
            else:
                processed_data[k] = v
        processed_datas.append(processed_data)

    # 保存到文件
    with open(save_path, "wb") as f:
        pickle.dump(processed_datas, f)
    print(f"数据已保存到: {save_path}")


# 读取数据
def load_data(load_path):
    # 读取文件
    with open(load_path, "rb") as f:
        processed_datas = pickle.load(f)

    # 转换回tensor
    datas = []
    for processed_data in processed_datas:
        data = {}
        for k, v in processed_data.items():
            if isinstance(v, np.ndarray):
                data[k] = torch.from_numpy(v)
            else:
                data[k] = v
        datas.append(data)
    print(f"已从 {load_path} 读取数据")
    return datas


save_path = (
    f"cached_data_{config.IMAGE_SIZE}_{config.DATASET.MGDPT_COARSE_SCALE}.pkl"
    if config.DATASET.DATA_SOURCE.lower() != "scannet"
    else "cached_scannet.pkl"
)

if save_path not in os.listdir():
    # 保存
    data_module = RCRM_Dataset(config=config)
    data_module.setup(stage="fit")
    train_dataloader = data_module.train_dataloader()
    val_dataloader = data_module.val_dataloader()

    datas = []

    for data in train_dataloader:
        datas.append(data)
        if len(datas) == 20:
            break
    save_data(datas, save_path)

# 读取
datas = load_data(save_path)


Model Structure and Parameters:
------------------------------------------------------------------------------------------
feature_backbone:    1.20M parameters
coarse_encoder:    3.86M parameters
recurrent_refine_unit:  816.17K parameters
------------------------------------------------------------------------------------------
Total Parameters: 5.87M



Loading MegaDepth dataset for validating: 100%|██████████| 5/5 [00:00<00:00, 125.88it/s]


数据已保存到: cached_data_640_8.pkl
已从 cached_data_640_8.pkl 读取数据


In [3]:
datas[0]["image0"].shape

torch.Size([1, 1, 640, 640])

In [4]:
datas[0]["image0"].max()

tensor(1.)

In [12]:
import torchvision
from torchvision.io import write_png
write_png((datas[0]["image1"][0] * 255).to(torch.uint8), "image1.png")


In [4]:
def recursive_to(item, device):
    if isinstance(item, torch.Tensor):
        return item.to(device)
    elif isinstance(item, dict):
        return {key: recursive_to(value, device) for key, value in item.items()}
    elif isinstance(item, list):
        return [recursive_to(value, device) for value in item]
    elif isinstance(item, tuple):
        return tuple(recursive_to(value, device) for value in item)
    else:
        return item

def compute_auc(errors, thresholds):
    """
    计算给定误差的 AUC（Area Under the Curve）
    
    Args:
        errors (torch.Tensor): 误差张量
        thresholds (list): AUC 计算的阈值列表
    
    Returns:
        dict: 不同阈值下的 AUC 值
    """
    errors = [0] + sorted(list(errors))
    recall = list(np.linspace(0, 1, len(errors)))

    aucs = []
    thresholds = [5, 10, 20]
    for thr in thresholds:
        last_index = np.searchsorted(errors, thr)
        y = recall[:last_index] + [recall[last_index - 1]]
        x = errors[:last_index] + [thr]
        aucs.append(np.trapz(y, x) / thr)

    return {f"auc@{t}": auc for t, auc in zip(thresholds, aucs)}

In [5]:
config.TRAINER.RANSAC_CONF = 0.99999
config.TRAINER.RANSAC_PIXEL_THR = 0.5

In [6]:
rcrm.max_coarse_matches = 4000
rcrm.max_intermediate_matches = 8000
rcrm.refine_iters = 4
rcrm.refine_lookup_radius = 3

In [7]:
rcrm.initial_forward()

In [ ]:
data = datas[4]
torch.cuda.empty_cache()
data = recursive_to(data, device)
compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE, config=config)
rcrm._forward_print_memory(data)
compute_supervision_fine(data, config)
torch.cuda.empty_cache()

In [ ]:
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [24]:
loss(data)

In [ ]:
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [26]:
_loss = data["loss"]
_loss.backward()

In [ ]:
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
before1 = torch.cuda.memory_allocated()

In [13]:
sim_matrix = data["sim_matrix"]
conf_matrix = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)

In [ ]:
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
after1 = torch.cuda.memory_allocated()

In [ ]:
(after1 - before1)/ 1024**3

In [ ]:
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
before2 = torch.cuda.memory_allocated()

In [20]:
conf = data["sim_matrix"]
conf_gt = data["conf_matrix_gt"]
b_pos, i_pos, j_pos = torch.where(conf_gt == 1)

b_pos_unique, inversed_b = b_pos.unique(sorted=True, return_inverse=True)
i_pos_unique, inversed_i = i_pos.unique(sorted=True, return_inverse=True)
j_pos_unique, inversed_j = j_pos.unique(sorted=True, return_inverse=True)

b = 0
row_pos = F.softmax(conf[b, i_pos_unique, :], 1)
col_pos = F.softmax(conf[b, :, j_pos_unique], 0)
pos_conf = (
    row_pos[inversed_i, j_pos_unique[inversed_j]]
    * col_pos[i_pos_unique[inversed_i], inversed_j]
)

In [ ]:
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
after2 = torch.cuda.memory_allocated()

In [ ]:
(after2 - before2)/ 1024**3

In [ ]:
for i in range(1):
    t_errs = []
    R_errs = []
    epi_errs = []
    for data in datas:
        data = recursive_to(data, device)
        compute_supervision_coarse(data, config.MODEL.COARSE_SCALE, config)
        with torch.no_grad():
            rcrm(data, training=True)
        compute_supervision_fine(data, config)
        
        b_idx_it = data["b_idx_it"]  # N
        offset_gt = data["coord_offset_gt"]  # N, 2
        scale1 = (
            data["scale1"][b_idx_it] * rcrm.fine_scale if "scale1" in data else rcrm.fine_scale
        )  # N
        fine_coord_1 = (offset_gt * rcrm.refine_lookup_radius * scale1) + data["intermediate_coord_1"]
        coord_0 = data["fine_coord_0"]
        coord_1 = fine_coord_1

        data.update(
            {
                "fine_coord_1": coord_1,
            }
        )


        # Errors
        compute_symmetrical_epipolar_errors(data)
        compute_pose_errors(data, config)
        t_errs.append(data["t_errs"])
        R_errs.append(data["R_errs"])
        epi_errs.append(data["epi_errs"])
        torch.cuda.empty_cache()
        # Print results
    t_errs = torch.tensor(t_errs).squeeze()
    R_errs = torch.tensor(R_errs).squeeze()
    aucs = [5, 10, 20]
    thres = 1e-4
    R_auc = compute_auc(R_errs, aucs)
    t_auc = compute_auc(t_errs, aucs)
    combined_errs = torch.max(R_errs, t_errs)
    combined_auc = compute_auc(combined_errs, aucs)
    print("\nR AUC:")
    for k, v in R_auc.items():
        print(f"{k}: {v:.4f}")
    print("\nt AUC")
    for k, v in t_auc.items():
        print(f"{k}: {v:.4f}")
    print("\nAUC:")
    for k, v in combined_auc.items():
        print(f"{k}: {v:.4f}")

    correct = 0
    total = 0
    for i in range(len(epi_errs)):
        correct += (epi_errs[i] < thres).sum()
        total += len(epi_errs[i])
    print(f"prec@{thres: e}: \t{((correct / total).item()) * 100 : .4f}")

# # 从 data 中获取图像并转换为 numpy 数组
# image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
# W1 = data["image0"].shape[-1]
# image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

# # 确保图像数据类型为 uint8
# image0 = image0.astype(np.uint8)
# image1 = image1.astype(np.uint8)

# # 将灰度图像转换为 BGR 图像
# image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
# image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

# # 将图像插值至两倍大小
# image0 = cv2.resize(
#     image0,
#     (image0.shape[1] * 2, image0.shape[0] * 2),
#     interpolation=cv2.INTER_LINEAR,
# )
# image1 = cv2.resize(
#     image1,
#     (image1.shape[1] * 2, image1.shape[0] * 2),
#     interpolation=cv2.INTER_LINEAR,
# )
# W1 *= 2

# # 将两张图像水平拼接在一起
# combined_image = cv2.hconcat([image0, image1])

# # 预定义一组常规的颜色
# colors = [
#     (255, 0, 0),  # 红色
#     (0, 255, 0),  # 绿色
#     (0, 0, 255),  # 蓝色
#     (255, 255, 0),  # 黄色
#     (255, 0, 255),  # 洋红
#     (0, 255, 255),  # 青色
#     (128, 0, 0),  # 深红
#     (0, 128, 0),  # 深绿
#     (0, 0, 128),  # 深蓝
#     (128, 128, 0),  # 橄榄绿
#     (128, 0, 128),  # 紫色
#     (0, 128, 128),  # 青绿色
#     (192, 192, 192),  # 银色
#     (128, 128, 128),  # 灰色
#     (0, 0, 0),  # 黑色
#     (255, 165, 0),  # 橙色
#     (255, 20, 147),  # 深粉色
#     (75, 0, 130),  # 靛蓝色
#     (240, 230, 140),  # 卡其色
#     (173, 216, 230),  # 淡蓝色
# ]

# # 在图像上绘制连线和圆圈
# lines = []
# circles = []

# # 每50个点画一次线
# for idx, ((x0, y0), (x1, y1)) in enumerate(zip(coord_0[::1], coord_1[::1])):
#     # 从预定义的颜色中随机选择一个颜色
#     color = (
#         np.random.randint(0, 256),
#         np.random.randint(0, 256),
#         np.random.randint(0, 256),
#     )

#     scale0 = data["scale0"][0] if "scale0" in data else 1
#     scale1 = data["scale1"][0] if "scale1" in data else 1

#     # 调整坐标至两倍大小
#     x0, y0, x1, y1 = (
#         x0 * 2 / scale0[0],
#         y0 * 2 / scale0[1],
#         x1 * 2 / scale1[0],
#         y1 * 2 / scale1[1],
#     )

#     # 存储线段信息
#     lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))

#     # 存储圆圈信息
#     circles.append(((int(x0), int(y0)), color))
#     circles.append(((int(x1) + W1, int(y1)), color))

# # 设置透明度 (0-1之间,1表示完全不透明)
# alpha = 0.5

# # 创建透明图层
# overlay = combined_image.copy()

# # 先在透明图层上绘制所有线段
# for start, end, color in lines:
#     cv2.line(overlay, start, end, color, 1, lineType=cv2.LINE_AA)

# # 再在透明图层上绘制所有圆圈
# for center, color in circles:
#     cv2.circle(overlay, center, 2, color, -1, lineType=cv2.LINE_AA)

# # 将带有所有线段和圆圈的图层与原图按透明度混合
# cv2.addWeighted(overlay, alpha, combined_image, 1 - alpha, 0, combined_image)

# # 使用 matplotlib 显示拼接后的图像
# plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
# plt.axis("off")  # 关闭坐标轴
# plt.show()

In [ ]:
import torch
import torch.nn.functional as F

# 创建一个简单的4x4图像张量
img = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8], 
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # 添加batch和channel维度 (1,1,4,4)

# 创建采样网格 - 我们想采样图像中心点(1.5, 1.5)
grid_align_corners = torch.tensor([[[[-1.0, -1.0]]]])  # align_corners=True时的归一化坐标
grid_no_align = torch.tensor([[[[-1.0, -1.0]]]])  # align_corners=False时的归一化坐标

# 使用grid_sample进行采样
sampled_align = F.grid_sample(img, grid_align_corners, align_corners=True)
sampled_no_align = F.grid_sample(img, grid_no_align, align_corners=False)

print("原始4x4图像:")
print(img[0,0])
print("\nalign_corners=True时采样结果:", sampled_align.item())
print("align_corners=False时采样结果:", sampled_no_align.item())


In [ ]:
for i in range(1):
    t_errs = []
    R_errs = []
    epi_errs = []

    for data in datas:
        data = recursive_to(data, device)
        compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
        scale0 = data["scale0"] if "scale0" in data else 1
        scale1 = data["scale1"] if "scale1" in data else 1

        with torch.no_grad():
            rcrm(data, training=True)

        compute_supervision_fine(data, config)

        data.update(
            {
                "fine_coord_1": data["fine_coord_1_gt"].clone().detach(),
            }
        )


        # Errors
        compute_symmetrical_epipolar_errors(data)
        compute_pose_errors(data, config)
        t_errs.append(data["t_errs"])
        R_errs.append(data["R_errs"])
        epi_errs.append(data["epi_errs"])
        torch.cuda.empty_cache()

        # # 从 data 中获取图像并转换为 numpy 数组
        # image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
        # W1 = data["image0"].shape[-1]
        # image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

        # # 确保图像数据类型为 uint8
        # image0 = image0.astype(np.uint8)
        # image1 = image1.astype(np.uint8)

        # # 将灰度图像转换为 BGR 图像
        # image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
        # image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

        # # 将图像插值至两倍大小
        # image0 = cv2.resize(
        #     image0,
        #     (image0.shape[1] * 2, image0.shape[0] * 2),
        #     interpolation=cv2.INTER_LINEAR,
        # )
        # image1 = cv2.resize(
        #     image1,
        #     (image1.shape[1] * 2, image1.shape[0] * 2),
        #     interpolation=cv2.INTER_LINEAR,
        # )
        # W1 *= 2

        # # 将两张图像水平拼接在一起
        # combined_image = cv2.hconcat([image0, image1])

        # # 预定义一组常规的颜色
        # colors = [
        #     (255, 0, 0),  # 红色
        #     (0, 255, 0),  # 绿色
        #     (0, 0, 255),  # 蓝色
        #     (255, 255, 0),  # 黄色
        #     (255, 0, 255),  # 洋红
        #     (0, 255, 255),  # 青色
        #     (128, 0, 0),  # 深红
        #     (0, 128, 0),  # 深绿
        #     (0, 0, 128),  # 深蓝
        #     (128, 128, 0),  # 橄榄绿
        #     (128, 0, 128),  # 紫色
        #     (0, 128, 128),  # 青绿色
        #     (192, 192, 192),  # 银色
        #     (128, 128, 128),  # 灰色
        #     (0, 0, 0),  # 黑色
        #     (255, 165, 0),  # 橙色
        #     (255, 20, 147),  # 深粉色
        #     (75, 0, 130),  # 靛蓝色
        #     (240, 230, 140),  # 卡其色
        #     (173, 216, 230),  # 淡蓝色
        # ]

        # # 在图像上绘制连线和圆圈
        # lines = []
        # circles = []

        # for idx, ((x0, y0), (x1, y1)) in enumerate(
        #     zip(fine_coord_0[mask], fine_coord_1[mask])
        # ):
        #     # 从预定义的颜色中随机选择一个颜色
        #     color = (
        #         np.random.randint(0, 256),
        #         np.random.randint(0, 256),
        #         np.random.randint(0, 256),
        #     )

        #     scale0 = data["scale0"][0] if "scale0" in data else 1
        #     scale1 = data["scale1"][0] if "scale1" in data else 1

        #     # 调整坐标至两倍大小
        #     x0, y0, x1, y1 = (
        #         x0 * 2 / scale0[0],
        #         y0 * 2 / scale0[1],
        #         x1 * 2 / scale1[0],
        #         y1 * 2 / scale1[1],
        #     )

        #     # 存储线段信息
        #     lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))

        #     # 存储圆圈信息
        #     circles.append(((int(x0), int(y0)), color))
        #     circles.append(((int(x1) + W1, int(y1)), color))

        # # 先绘制所有线段
        # for start, end, color in lines:
        #     cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

        # # 再绘制所有圆圈
        # for center, color in circles:
        #     cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

        # # 保存图像到文件
        # output_path = "sample_match.png"
        # cv2.imwrite(output_path, combined_image)

        # # 使用 matplotlib 显示拼接后的图像
        # plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
        # plt.axis("off")  # 关闭坐标轴
        # plt.show()
        # print(f"R_errs: {data['R_errs']}")
        # print(f"t_errs: {data['t_errs']}")

    # Print results
    t_errs = torch.tensor(t_errs).squeeze()
    R_errs = torch.tensor(R_errs).squeeze()
    aucs = [5, 10, 20]
    thres = 1e-4
    R_auc = compute_auc(R_errs, aucs)
    t_auc = compute_auc(t_errs, aucs)
    combined_errs = torch.max(R_errs, t_errs)
    combined_auc = compute_auc(combined_errs, aucs)
    print("\nR AUC:")
    for k, v in R_auc.items():
        print(f"{k}: {v:.4f}")
    print("\nt AUC")
    for k, v in t_auc.items():
        print(f"{k}: {v:.4f}")
    print("\nAUC:")
    for k, v in combined_auc.items():
        print(f"{k}: {v:.4f}")

    correct = 0
    total = 0
    for i in range(len(epi_errs)):
        correct += (epi_errs[i] < thres).sum()
        total += len(epi_errs[i])
    print(f"prec@{thres: e}: \t{((correct / total).item()) * 100 : .4f}")

In [ ]:
t_errs = []
R_errs = []
epi_errs = []

for data in datas:
    data = recursive_to(data, device)
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    W = data["image0"].shape[-1] / config.MODEL.COARSE_SCALE
    if config.MODEL.VERSION == "v1":
        scale = (
            1
            if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
            else config["MODEL"]["FINE_SCALE"]
        )
    elif config.MODEL.VERSION == "v2":
        scale = (
            1
            if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
            else config["MODEL"]["BACKBONE"]["RESOLUTION"][0]
        )
    scale0 = data["scale0"] if "scale0" in data else 1
    scale1 = data["scale1"] if "scale1" in data else 1
    coarse_coord_0 = torch.tensor(
        [[i % W, i // W] for i in data["spv_i_ids"]], device=device
    )
    coarse_coord_1 = torch.tensor(
        [[i % W, i // W] for i in data["spv_j_ids"]], device=device
    )

    data.update(
        {
            "b_idx_c": data["spv_b_ids"],
            "i_idx_c": data["spv_i_ids"],
            "j_idx_c": data["spv_j_ids"],
        }
    )

    with torch.no_grad():
        rcrm(data, training=True)

    edited_compute_supervision_fine(data, config)

    coarse_coord_0 = data["coarse_coord_0"]
    coarse_coord_1 = data["coarse_coord_1"]
    fine_coord_0 = data["fine_coord_0"].clone().detach()
    fine_coord_1 = data["fine_coord_1"].clone().detach()
    # 添加固定扰动
    fixed_noise = torch.tensor([[0, 0]], device=device).repeat(
        data["coord_offset_f_gt"].shape[0], 1
    )
    radius = (
        config["MODEL"]["COARSE_SCALE"] / 2
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else int(config["MODEL"]["COARSE_SCALE"] // config["MODEL"]["FINE_SCALE"]) / 2
    )
    mask = torch.linalg.norm(data["coord_offset_f_gt"], ord=float("inf"), dim=1) < 2.0
    data["coord_offset_f_gt"][~mask] = 0
    fine_coord_1 = (
        coarse_coord_1
        + (data["coord_offset_f_gt"] + fixed_noise) * scale1 * scale * 4.0
    )

    data.update(
        {
            "fine_coord_0": fine_coord_0[mask],
            "fine_coord_1": fine_coord_1[mask],
            "b_idx_c": data["b_idx_c"][mask],
        }
    )

    # Errors
    compute_symmetrical_epipolar_errors(data)
    compute_pose_errors(data, config)
    t_errs.append(data["t_errs"])
    R_errs.append(data["R_errs"])
    epi_errs.append(data["epi_errs"])
    torch.cuda.empty_cache()

    # # 从 data 中获取图像并转换为 numpy 数组
    # image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
    # W1 = data["image0"].shape[-1]
    # image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

    # # 确保图像数据类型为 uint8
    # image0 = image0.astype(np.uint8)
    # image1 = image1.astype(np.uint8)

    # # 将灰度图像转换为 BGR 图像
    # image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
    # image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

    # # 将图像插值至两倍大小
    # image0 = cv2.resize(
    #     image0,
    #     (image0.shape[1] * 2, image0.shape[0] * 2),
    #     interpolation=cv2.INTER_LINEAR,
    # )
    # image1 = cv2.resize(
    #     image1,
    #     (image1.shape[1] * 2, image1.shape[0] * 2),
    #     interpolation=cv2.INTER_LINEAR,
    # )
    # W1 *= 2

    # # 将两张图像水平拼接在一起
    # combined_image = cv2.hconcat([image0, image1])

    # # 预定义一组常规的颜色
    # colors = [
    #     (255, 0, 0),  # 红色
    #     (0, 255, 0),  # 绿色
    #     (0, 0, 255),  # 蓝色
    #     (255, 255, 0),  # 黄色
    #     (255, 0, 255),  # 洋红
    #     (0, 255, 255),  # 青色
    #     (128, 0, 0),  # 深红
    #     (0, 128, 0),  # 深绿
    #     (0, 0, 128),  # 深蓝
    #     (128, 128, 0),  # 橄榄绿
    #     (128, 0, 128),  # 紫色
    #     (0, 128, 128),  # 青绿色
    #     (192, 192, 192),  # 银色
    #     (128, 128, 128),  # 灰色
    #     (0, 0, 0),  # 黑色
    #     (255, 165, 0),  # 橙色
    #     (255, 20, 147),  # 深粉色
    #     (75, 0, 130),  # 靛蓝色
    #     (240, 230, 140),  # 卡其色
    #     (173, 216, 230),  # 淡蓝色
    # ]

    # # 在图像上绘制连线和圆圈
    # lines = []
    # circles = []

    # for idx, ((x0, y0), (x1, y1)) in enumerate(
    #     zip(fine_coord_0[mask], fine_coord_1[mask])
    # ):
    #     # 从预定义的颜色中随机选择一个颜色
    #     color = (
    #         np.random.randint(0, 256),
    #         np.random.randint(0, 256),
    #         np.random.randint(0, 256),
    #     )

    #     scale0 = data["scale0"][0] if "scale0" in data else 1
    #     scale1 = data["scale1"][0] if "scale1" in data else 1

    #     # 调整坐标至两倍大小
    #     x0, y0, x1, y1 = (
    #         x0 * 2 / scale0[0],
    #         y0 * 2 / scale0[1],
    #         x1 * 2 / scale1[0],
    #         y1 * 2 / scale1[1],
    #     )

    #     # 存储线段信息
    #     lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))

    #     # 存储圆圈信息
    #     circles.append(((int(x0), int(y0)), color))
    #     circles.append(((int(x1) + W1, int(y1)), color))

    # # 先绘制所有线段
    # for start, end, color in lines:
    #     cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

    # # 再绘制所有圆圈
    # for center, color in circles:
    #     cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

    # # 保存图像到文件
    # output_path = "sample_match.png"
    # cv2.imwrite(output_path, combined_image)

    # # 使用 matplotlib 显示拼接后的图像
    # plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
    # plt.axis("off")  # 关闭坐标轴
    # plt.show()
    # print(f"R_errs: {data['R_errs']}")
    # print(f"t_errs: {data['t_errs']}")

# Print results
t_errs = torch.tensor(t_errs).squeeze()
R_errs = torch.tensor(R_errs).squeeze()
aucs = [5, 10, 20]
thres = 1e-4
R_auc = compute_auc(R_errs, aucs)
t_auc = compute_auc(t_errs, aucs)
combined_errs = torch.max(R_errs, t_errs)
combined_auc = compute_auc(combined_errs, aucs)
print("\nR AUC:")
for k, v in R_auc.items():
    print(f"{k}: {v:.4f}")
print("\nt AUC")
for k, v in t_auc.items():
    print(f"{k}: {v:.4f}")
print("\nAUC:")
for k, v in combined_auc.items():
    print(f"{k}: {v:.4f}")


correct = 0
total = 0
for i in range(len(epi_errs)):
    correct += (epi_errs[i] < thres).sum()
    total += len(epi_errs[i])
print(f"prec@{thres: e}: \t{((correct / total).item()) * 100 : .4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

config = get_cfg_defaults()
config.IMAGE_SIZE = config.MODEL.BACKBONE.INPUT_SIZE = (
    config.DATASET.MGDPT_IMG_RESIZE
) = 640
config.DEVICE.GPU_IDX = "1,"
setup_gpus(config.DEVICE.GPU_IDX)


class OffsetPredictor(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv_offset = nn.Sequential(
            nn.Conv2d(2 * in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, int(in_channels // 2), kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(int(in_channels // 2), 2, kernel_size=3, padding=1),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, feat0, feat1):
        concat_feat = torch.cat([feat0, feat1], dim=1)
        return self.conv_offset(concat_feat)


def generate_random_feature_maps(batch_size=8, channels=64, size=40):
    """生成更有结构的特征图"""
    # 生成基础随机特征
    feat = torch.randn(batch_size, channels, size, size)

    # 添加空间结构（例如使用高斯滤波）
    kernel_size = 3
    sigma = 1.0
    gaussian = (
        torch.tensor(
            [[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]], device=feat.device
        )
        / 16.0
    )

    gaussian = gaussian.view(1, 1, kernel_size, kernel_size)
    gaussian = gaussian.repeat(channels, 1, 1, 1)

    # 添加空间平滑
    feat = F.conv2d(feat, gaussian, padding=kernel_size // 2, groups=channels)

    return feat


def warp_features(feat, offset_x, offset_y):
    """使用给定的偏移量对特征图进行变形"""
    B, C, H, W = feat.shape
    device = feat.device

    # 创建基础网格
    y_grid, x_grid = torch.meshgrid(
        torch.arange(H, device=device), torch.arange(W, device=device), indexing="ij"
    )

    # 添加偏移量
    x_grid = x_grid.float() + offset_x
    y_grid = y_grid.float() + offset_y

    # 归一化坐标到 [-1, 1]
    x_grid = 2.0 * x_grid / (W - 1) - 1.0
    y_grid = 2.0 * y_grid / (H - 1) - 1.0

    # 组合网格
    grid = torch.stack([x_grid, y_grid], dim=-1)
    grid = grid.expand(B, H, W, 2)

    # 进行网格采样
    warped_feat = F.grid_sample(
        feat, grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )

    return warped_feat


def extract_window(feat, center_h, center_w, window_size=7):
    """从特征图中提取窗口"""
    half_size = window_size // 2
    return feat[
        :,
        :,
        center_h - half_size : center_h + half_size,
        center_w - half_size : center_w + half_size,
    ]


# 训练设置
def train_offset_predictor(model, optimizer, device):
    # 超参数
    batch_size = 16
    channels = 256
    feat_size = 40
    window_size = 4
    n_epochs = 100

    # 训练循环
    for epoch in range(n_epochs):
        total_loss = 0
        for _ in range(10):  # 每个epoch的批次数
            # 生成随机特征图
            feat0 = generate_random_feature_maps(batch_size, channels, feat_size).to(
                device
            )

            # 生成随机偏移量（作为ground truth）
            max_offset = 0.5
            gt_offset_x = (torch.rand(1) * 2 - 1) * max_offset  # 已经在 [-1, 1] 范围内
            gt_offset_y = (torch.rand(1) * 2 - 1) * max_offset  # 已经在 [-1, 1] 范围内

            # 将归一化的偏移量转换为实际像素偏移（用于warping）
            pixel_offset_x = gt_offset_x * (window_size // 2)  # 转换为实际像素偏移
            pixel_offset_y = gt_offset_y * (window_size // 2)  # 转换为实际像素偏移

            # 对第二个特征图进行warp（使用像素偏移）
            feat1 = warp_features(feat0, pixel_offset_x.item(), pixel_offset_y.item())

            # 提取中心窗口
            center_h = feat_size // 2
            center_w = feat_size // 2
            feat0_window = extract_window(feat0, center_h, center_w, window_size)
            feat1_window = extract_window(feat1, center_h, center_w, window_size)

            # 预测偏移量
            pred_offset = model(feat0_window, feat1_window)

            # 计算损失
            gt_offset = (
                torch.tensor([[gt_offset_x, gt_offset_y]], device=device)
                .view(1, 2, 1, 1)
                .expand(-1, -1, window_size, window_size)
            )

            loss = F.mse_loss(pred_offset, gt_offset.expand(batch_size, -1, -1, -1))

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # 打印训练进度
        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / 10
            print(f"Epoch [{epoch+1}/{n_epochs}], Average Loss: {avg_loss:.4f}")
            print(f"GT offset: ({gt_offset_x.item():.2f}, {gt_offset_y.item():.2f})")
            print(
                f"Pred offset: ({pred_offset[0,0,window_size//2,window_size//2].item():.2f}, "
                f"{pred_offset[0,1,window_size//2,window_size//2].item():.2f})"
            )
            print("", flush=True)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = OffsetPredictor(256).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_offset_predictor(model, optimizer, device)

In [ ]:
for param_group in optimizer.param_groups:
    param_group["lr"] = 1e-4
train_offset_predictor(model, optimizer, device)

In [2]:
latest_ckpt_path = (
    "logs/tb_logs/MegaDepth_840_v1_8_2_VMamba_T_cropped_C2F4_I4R3_A/version_0/"
)
latest_ckpt = "checkpoints/epoch=8-auc@5=0.304-auc@10=0.436-auc@20=0.562.ckpt"
sys.path.append(latest_ckpt_path)
get_cfg_defaults = importlib.import_module("config").get_cfg_defaults

In [3]:
config = get_cfg_defaults()
config.DEVICE.GPU_IDX = "7,"

ransac_thres = 0.2
ransac_times = 1
coarse_thres = 0.2
intermediate_thres = 0.06
coarse_max = 8000
intermediate_max = 16000
refine_iters = 2
image_size = 1200

config.TRAINER.RANSAC_PIXEL_THR = ransac_thres
config.TRAINER.RANSAC_TIMES = ransac_times
config.MODEL.COARSE_MATCHING.THRESHOLD = coarse_thres
config.MODEL.INTERMEDIATE_MATCHING.THRESHOLD = intermediate_thres
config.IMAGE_SIZE = config.MODEL.BACKBONE.INPUT_SIZE = config.DATASET.MGDPT_IMG_RESIZE = image_size
config.MODEL.COARSE_MATCHING.MAX_MATCHES = coarse_max
config.MODEL.INTERMEDIATE_MATCHING.MAX_MATCHES = intermediate_max
config.MODEL.REFINE_ITERS = refine_iters

In [ ]:
config.DEVICE.ENABLE_DDP = False
# setup exact gpus available and set CUDA_VISIBLE_DEVICES variable
n_gpu_available = (
    setup_gpus(config.DEVICE.GPU_IDX) if config.DEVICE.ENABLE_GPU else 0
)
config.TRAINER.WORLD_SIZE = n_gpu_available * config.DEVICE.NUM_NODES
config.TRAINER.TRUE_BATCH_SIZE = (
    config.TRAINER.WORLD_SIZE * config.LOADER.BATCH_SIZE
)
config.TRAINER.SCALING = (
    config.TRAINER.TRUE_BATCH_SIZE / config.TRAINER.CANONICAL_BS
)
config.TRAINER.TRUE_LR = config.TRAINER.CANONICAL_LR * config.TRAINER.SCALING
config.TRAINER.WARMUP_STEP = math.floor(
    config.TRAINER.WARMUP_STEP / config.TRAINER.SCALING
)
device = torch.device("cuda:0")

if config["MODEL"]["VERSION"] == "v1":
    _RCRM = RCRM_v1

rcrm = _RCRM(config=config["MODEL"])
print_params_summary(rcrm)
rcrm.to(device)
loss = RCRM_Loss(config.LOSS)
loss.to(device)

# 保存数据
def save_data(datas, save_path):
    # 转换数据
    processed_datas = []
    for data in datas:
        processed_data = {}
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                processed_data[k] = v.cpu().numpy()
            else:
                processed_data[k] = v
        processed_datas.append(processed_data)
    
    # 保存到文件
    with open(save_path, 'wb') as f:
        pickle.dump(processed_datas, f)
    print(f"数据已保存到: {save_path}")

# 读取数据
def load_data(load_path):
    # 读取文件
    with open(load_path, 'rb') as f:
        processed_datas = pickle.load(f)
    
    # 转换回tensor
    datas = []
    for processed_data in processed_datas:
        data = {}
        for k, v in processed_data.items():
            if isinstance(v, np.ndarray):
                data[k] = torch.from_numpy(v)
            else:
                data[k] = v
        datas.append(data)
    print(f"已从 {load_path} 读取数据")
    return datas

save_path = f"cached_data_{config.IMAGE_SIZE}_{config.DATASET.MGDPT_COARSE_SCALE}.pkl"

# # 保存
# data_module = RCRM_Dataset(config=config)
# data_module.setup(stage="fit")
# train_dataloader = data_module.train_dataloader()
# val_dataloader = data_module.val_dataloader()

# datas = []

# for data in train_dataloader:
#     datas.append(data)
#     if len(datas) == 20:
#         break
# save_data(datas, save_path)

# 读取
datas = load_data(save_path)

In [ ]:
rcrm.load_state_dict(torch.load(latest_ckpt_path+"/"+latest_ckpt)["state_dict"])
rcrm.to(device)

In [6]:
def recursive_to(item, device):
    if isinstance(item, torch.Tensor):
        return item.to(device)
    elif isinstance(item, dict):
        return {key: recursive_to(value, device) for key, value in item.items()}
    elif isinstance(item, list):
        return [recursive_to(value, device) for value in item]
    elif isinstance(item, tuple):
        return tuple(recursive_to(value, device) for value in item)
    else:
        return item

In [7]:
config.TRAINER.RANSAC_CONF = 0.99999
config.TRAINER.RANSAC_PIXEL_THR = 0.2

In [8]:
def compute_auc(errors, thresholds):
    """
    计算给定误差的 AUC（Area Under the Curve）
    
    Args:
        errors (torch.Tensor): 误差张量
        thresholds (list): AUC 计算的阈值列表
    
    Returns:
        dict: 不同阈值下的 AUC 值
    """
    errors = errors.cpu().numpy()
    sort_idx = np.argsort(errors)
    errors = errors[sort_idx]
    recall = (np.arange(len(errors)) + 1) / len(errors)
    errors = np.r_[0., errors]
    recall = np.r_[0., recall]
    
    aucs = {}
    for t in thresholds:
        last_index = np.searchsorted(errors, t)
        r = np.r_[recall[:last_index], recall[last_index-1]]
        e = np.r_[errors[:last_index], t]
        aucs[f'auc@{t}'] = np.trapz(r, x=e)/t
    
    return aucs

In [30]:
rcrm.coarse_match_thres = 0.25
rcrm.intermediate_match_thres = 0.1
data = datas[10]
data = recursive_to(data, device)
with torch.no_grad():
    rcrm.forward(data, training=False)


In [ ]:
fine_coord_0 = data["fine_coord_0"]
fine_coord_1 = data["fine_coord_1"]

# 从 data 中获取图像并转换为 numpy 数组
image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
W1 = data["image0"].shape[-1] 
image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

# 确保图像数据类型为 uint8
image0 = image0.astype(np.uint8)
image1 = image1.astype(np.uint8)

# 将灰度图像转换为 BGR 图像
image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

# 将图像插值至两倍大小
image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
W1 *= 2

# 将两张图像水平拼接在一起
combined_image = cv2.hconcat([image0, image1])

# 预定义一组常规的颜色
colors = [
    (255, 0, 0),    # 红色
    (0, 255, 0),    # 绿色
    (0, 0, 255),    # 蓝色
    (255, 255, 0),  # 黄色
    (255, 0, 255),  # 洋红
    (0, 255, 255),  # 青色
    (128, 0, 0),    # 深红
    (0, 128, 0),    # 深绿
    (0, 0, 128),    # 深蓝
    (128, 128, 0),  # 橄榄绿
    (128, 0, 128),  # 紫色
    (0, 128, 128),  # 青绿色
    (192, 192, 192),# 银色
    (128, 128, 128),# 灰色
    (0, 0, 0),      # 黑色
    (255, 165, 0),  # 橙色
    (255, 20, 147), # 深粉色
    (75, 0, 130),   # 靛蓝色
    (240, 230, 140),# 卡其色
    (173, 216, 230) # 淡蓝色
]

# 在图像上绘制连线和圆圈
lines = []
circles = []

for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    # 从预定义的颜色中随机选择一个颜色
    color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
    
    scale0 = data["scale0"][0] if "scale0" in data else 1
    scale1 = data["scale1"][0] if "scale1" in data else 1
    
    # 调整坐标至两倍大小
    x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
    
    # 存储线段信息
    lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
    
    # 存储圆圈信息
    circles.append(((int(x0), int(y0)), color))
    circles.append(((int(x1) + W1, int(y1)), color))

# 先绘制所有线段
for start, end, color in lines:
    cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

# 再绘制所有圆圈
for center, color in circles:
    cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

# 保存图像到文件
# output_path = 'sample_match.png'
# cv2.imwrite(output_path, combined_image)

# 使用 matplotlib 显示拼接后的图像
plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
plt.axis('off')  # 关闭坐标轴
plt.show()    

In [ ]:
for data in datas:
    data = recursive_to(data, device)
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    W = data["image0"].shape[-1] / config.MODEL.COARSE_SCALE
    scale = (
        1
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else config["MODEL"]["FINE_SCALE"]
    )
    scale0 = data["scale0"] if "scale0" in data else 1
    scale1 = data["scale1"] if "scale1" in data else 1
    # coarse_coord_0 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_j_ids"]], device=device)
    coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["spv_i_ids"]], device=device)
    coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["spv_j_ids"]], device=device)
    
    data.update({
        "b_idx_c": data["spv_b_ids"],
        "i_idx_c": data["spv_i_ids"],
        "j_idx_c": data["spv_j_ids"],
    })
    
    compute_supervision_fine(data, config)
    
    fine_coord_0 = coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    # 添加随机扰动
    random_noise = torch.randn_like(data["coord_offset_f_gt"], device=device) * 0.  # 0.1 是扰动的强度，可以根据需要调整
    fixed_noise = torch.tensor([[0.0, 0.0]], device=device).repeat(data["coord_offset_f_gt"].shape[0],1)
    radius = (
        config["MODEL"]["COARSE_SCALE"] // 2
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else int(config["MODEL"]["COARSE_SCALE"] // config["MODEL"]["FINE_SCALE"]) // 2
    )
    mask = torch.linalg.norm(data["coord_offset_f_gt"], ord=float("inf"), dim=1) < 1.0
    data["coord_offset_f_gt"][~mask] = 0
    fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["coord_offset_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius

    data.update({"coarse_coord_0": coarse_coord_0, "coarse_coord_1": coarse_coord_1, "fine_coord_0": fine_coord_0, "fine_coord_1": fine_coord_1})

    # 从 data 中获取图像并转换为 numpy 数组
    image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
    W1 = data["image0"].shape[-1] 
    image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

    # 确保图像数据类型为 uint8
    image0 = image0.astype(np.uint8)
    image1 = image1.astype(np.uint8)

    # 将灰度图像转换为 BGR 图像
    image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
    image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

    # 将图像插值至两倍大小
    image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    W1 *= 2

    # 将两张图像水平拼接在一起
    combined_image = cv2.hconcat([image0, image1])

    # 预定义一组常规的颜色
    colors = [
        (255, 0, 0),    # 红色
        (0, 255, 0),    # 绿色
        (0, 0, 255),    # 蓝色
        (255, 255, 0),  # 黄色
        (255, 0, 255),  # 洋红
        (0, 255, 255),  # 青色
        (128, 0, 0),    # 深红
        (0, 128, 0),    # 深绿
        (0, 0, 128),    # 深蓝
        (128, 128, 0),  # 橄榄绿
        (128, 0, 128),  # 紫色
        (0, 128, 128),  # 青绿色
        (192, 192, 192),# 银色
        (128, 128, 128),# 灰色
        (0, 0, 0),      # 黑色
        (255, 165, 0),  # 橙色
        (255, 20, 147), # 深粉色
        (75, 0, 130),   # 靛蓝色
        (240, 230, 140),# 卡其色
        (173, 216, 230) # 淡蓝色
    ]

    # 在图像上绘制连线和圆圈
    lines = []
    circles = []

    for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
        # 从预定义的颜色中随机选择一个颜色
        color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
        
        scale0 = data["scale0"][0] if "scale0" in data else 1
        scale1 = data["scale1"][0] if "scale1" in data else 1
        
        # 调整坐标至两倍大小
        x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
        
        # 存储线段信息
        lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
        
        # 存储圆圈信息
        circles.append(((int(x0), int(y0)), color))
        circles.append(((int(x1) + W1, int(y1)), color))

    # 先绘制所有线段
    for start, end, color in lines:
        cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

    # 再绘制所有圆圈
    for center, color in circles:
        cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

    # 保存图像到文件
    output_path = 'sample_match.png'
    cv2.imwrite(output_path, combined_image)

    # 使用 matplotlib 显示拼接后的图像
    plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
    plt.axis('off')  # 关闭坐标轴
    plt.show()    

In [ ]:
t_errs = []
R_errs = []
epi_errs = []
fine_coord_0s = []
fine_coord_1s = []
coarse_coord_1s = []
target_fine_coord_0s = []
target_fine_coord_1s = []
target_coarse_coord_1s = []

for data in datas:
    data = recursive_to(data, device)
    with torch.no_grad():
        rcrm(data)
        torch.cuda.empty_cache()
    coarse_coord_0 = data["coarse_coord_0"]
    coarse_coord_1 = data["coarse_coord_1"]
    fine_coord_0 = data["fine_coord_0"].clone().detach()
    fine_coord_1 = data["fine_coord_1"].clone().detach()
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    W = data["image0"].shape[-1] / config.MODEL.COARSE_SCALE
    scale = (
        1
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else config["MODEL"]["FINE_SCALE"]
    )
    scale0 = data["scale0"] if "scale0" in data else 1
    scale1 = data["scale1"] if "scale1" in data else 1
    # coarse_coord_0 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_j_ids"]], device=device)
    # coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["spv_j_ids"]], device=device)
    target_coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["i_idx_c"]], device=device)
    target_coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["j_idx_c"]], device=device)
    
    # data.update({
    #     "b_idx_c": data["spv_b_ids"],
    #     "i_idx_c": data["spv_i_ids"],
    #     "j_idx_c": data["spv_j_ids"],
    # })
    
    compute_supervision_fine(data, config)
    
    # fine_coord_0 = coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    target_fine_coord_0 = target_coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    # 添加随机扰动
    random_noise = torch.randn_like(data["expec_f_gt"], device=device) * 0.  # 0.1 是扰动的强度，可以根据需要调整
    fixed_noise = torch.tensor([[0.0, 0.0]], device=device).repeat(data["expec_f_gt"].shape[0],1)
    radius = (
        config["MODEL"]["COARSE_SCALE"] // 2
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else int(config["MODEL"]["COARSE_SCALE"] // config["MODEL"]["FINE_SCALE"]) // 2
    )
    mask = torch.linalg.norm(data["expec_f_gt"], ord=float("inf"), dim=1) < 1.0
    data["expec_f_gt"][~mask] = 0
    # fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["expec_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius
    target_fine_coord_1 = target_coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["expec_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius
    
    fine_coord_0s.append(fine_coord_0[mask])
    fine_coord_1s.append(fine_coord_1[mask])
    target_fine_coord_0s.append(target_fine_coord_0[mask])
    target_fine_coord_1s.append(target_fine_coord_1[mask])
    coarse_coord_1s.append(coarse_coord_1[mask])
    target_coarse_coord_1s.append(target_coarse_coord_1[mask])
    # fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1
    # data.update({"coarse_coord_0": coarse_coord_0, "coarse_coord_1": coarse_coord_1, "fine_coord_0": fine_coord_0, "fine_coord_1": fine_coord_1})

    # # 从 data 中获取图像并转换为 numpy 数组
    # image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
    # W1 = data["image0"].shape[-1] 
    # image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

    # # 确保图像数据类型为 uint8
    # image0 = image0.astype(np.uint8)
    # image1 = image1.astype(np.uint8)

    # # 将灰度图像转换为 BGR 图像
    # image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
    # image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

    # # 将图像插值至两倍大小
    # image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    # image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    # W1 *= 2

    # # 将两张图像水平拼接在一起
    # combined_image = cv2.hconcat([image0, image1])

    # # 预定义一组常规的颜色
    # colors = [
    #     (255, 0, 0),    # 红色
    #     (0, 255, 0),    # 绿色
    #     (0, 0, 255),    # 蓝色
    #     (255, 255, 0),  # 黄色
    #     (255, 0, 255),  # 洋红
    #     (0, 255, 255),  # 青色
    #     (128, 0, 0),    # 深红
    #     (0, 128, 0),    # 深绿
    #     (0, 0, 128),    # 深蓝
    #     (128, 128, 0),  # 橄榄绿
    #     (128, 0, 128),  # 紫色
    #     (0, 128, 128),  # 青绿色
    #     (192, 192, 192),# 银色
    #     (128, 128, 128),# 灰色
    #     (0, 0, 0),      # 黑色
    #     (255, 165, 0),  # 橙色
    #     (255, 20, 147), # 深粉色
    #     (75, 0, 130),   # 靛蓝色
    #     (240, 230, 140),# 卡其色
    #     (173, 216, 230) # 淡蓝色
    # ]

    # # 在图像上绘制连线和圆圈
    # lines = []
    # circles = []

    # for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    #     # 从预定义的颜色中随机选择一个颜色
    #     color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
        
    #     scale0 = data["scale0"][0] if "scale0" in data else 1
    #     scale1 = data["scale1"][0] if "scale1" in data else 1
        
    #     # 调整坐标至两倍大小
    #     x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
        
    #     # 存储线段信息
    #     lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
        
    #     # 存储圆圈信息
    #     circles.append(((int(x0), int(y0)), color))
    #     circles.append(((int(x1) + W1, int(y1)), color))

    # # 先绘制所有线段
    # for start, end, color in lines:
    #     cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

    # # 再绘制所有圆圈
    # for center, color in circles:
    #     cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

    # # 保存图像到文件
    # output_path = 'sample_match.png'
    # cv2.imwrite(output_path, combined_image)

    # # 使用 matplotlib 显示拼接后的图像
    # plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
    # plt.axis('off')  # 关闭坐标轴
    # plt.show()    
    
    b_idx_c = torch.zeros(len(fine_coord_0), dtype=torch.int32)
    data.update({"b_idx_c": b_idx_c})
    compute_symmetrical_epipolar_errors(data)
    compute_pose_errors(data, config)
    # print(f"t_errs: {data['t_errs']}\nR_errs: {data['R_errs']}")
    t_errs.append(data['t_errs'])
    R_errs.append(data['R_errs'])
    epi_errs.append(data['epi_errs'])
    torch.cuda.empty_cache()

# Print results
t_errs = torch.tensor(t_errs).squeeze()
R_errs = torch.tensor(R_errs).squeeze()
aucs = [5, 10, 20]
thres = 1e-4
R_auc = compute_auc(R_errs, aucs)
t_auc = compute_auc(t_errs, aucs)
combined_errs = torch.max(R_errs, t_errs)
combined_auc = compute_auc(combined_errs, aucs)
print("\nR AUC:")
for k, v in R_auc.items():
    print(f"{k}: {v:.4f}")
print("\nt AUC")
for k, v in t_auc.items():
    print(f"{k}: {v:.4f}")
print("\nAUC:")
for k, v in combined_auc.items():
    print(f"{k}: {v:.4f}")


correct = 0
total = 0
for i in range(len(epi_errs)):
    correct += (epi_errs[i] < thres).sum()
    total += len(epi_errs[i])
print(f"prec@{thres: e}: \t{((correct / total).item()) * 100 : .4f}")

In [ ]:
fine_coord_0s = torch.concat(fine_coord_0s, dim=0)
fine_coord_1s = torch.concat(fine_coord_1s, dim=0)
coarse_coord_1s = torch.concat(coarse_coord_1s, dim=0)
target_fine_coord_0s = torch.concat(target_fine_coord_0s, dim=0)
target_fine_coord_1s = torch.concat(target_fine_coord_1s, dim=0)
target_coarse_coord_1s = torch.concat(target_coarse_coord_1s, dim=0)

In [ ]:
diff = (fine_coord_1s - coarse_coord_1s)
diff_x = diff[:,0] ** 2
diff_y = diff[:,1] ** 2
print(f"diff_x: {diff_x.mean()}\ndiff_y: {diff_y.mean()}")

In [ ]:
diff = (fine_coord_1s - target_fine_coord_1s)
diff_x = diff[:,0] ** 2
diff_y = diff[:,1] ** 2
print(f"diff_x: {diff_x.mean()}\ndiff_y: {diff_y.mean()}")

In [ ]:
data["expec_f"][0]

In [ ]:
with torch.no_grad():
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    rcrm(data)
    # compute epi_errs for each match
    compute_symmetrical_epipolar_errors(data)
    # compute R_errs, t_errs, pose_errs for each pair
    compute_pose_errors(data, config)
    rel_pair_names = list(zip(*data["pair_names"]))
    bs = data["image0"].size(0)
    metrics = {
        # to filter duplicate pairs caused by DistributedSampler
        "identifiers": ["#".join(rel_pair_names[b]) for b in range(bs)],
        "epi_errs": [
            data["epi_errs"][data["b_idx_c"] == b].cpu().numpy() for b in range(bs)
        ],
        "R_errs": data["R_errs"],
        "t_errs": data["t_errs"],
        "inliers": data["inliers"],
    }
    val_metrics_4tb = aggregate_metrics(metrics, 1e-4)

In [ ]:
print(f"t_errs: {data['t_errs']}\nR_errs: {data['R_errs']}\nInliers: {data['inliers']}")

In [ ]:
coarse_coord_0 = data["coarse_coord_0"]
coarse_coord_1 = data["coarse_coord_1"]
fine_coord_0 = data["fine_coord_0"]
fine_coord_1 = data["fine_coord_1"]

In [ ]:
# 从 data 中获取图像并转换为 numpy 数组
image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
W1 = data["hw0_i"][1]
image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

# 确保图像数据类型为 uint8
image0 = image0.astype(np.uint8)
image1 = image1.astype(np.uint8)

# 将灰度图像转换为 BGR 图像
image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

# 将图像插值至两倍大小
image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
W1 *= 2

# 将两张图像水平拼接在一起
combined_image = cv2.hconcat([image0, image1])

# 预定义一组常规的颜色
colors = [
    (255, 0, 0),    # 红色
    (0, 255, 0),    # 绿色
    (0, 0, 255),    # 蓝色
    (255, 255, 0),  # 黄色
    (255, 0, 255),  # 洋红
    (0, 255, 255),  # 青色
    (128, 0, 0),    # 深红
    (0, 128, 0),    # 深绿
    (0, 0, 128),    # 深蓝
    (128, 128, 0),  # 橄榄绿
    (128, 0, 128),  # 紫色
    (0, 128, 128),  # 青绿色
    (192, 192, 192),# 银色
    (128, 128, 128),# 灰色
    (0, 0, 0),      # 黑色
    (255, 165, 0),  # 橙色
    (255, 20, 147), # 深粉色
    (75, 0, 130),   # 靛蓝色
    (240, 230, 140),# 卡其色
    (173, 216, 230) # 淡蓝色
]

# 在图像上绘制连线和圆圈
lines = []
circles = []

for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    if data["inliers"][0][idx] == 0:
        continue
    # 从预定义的颜色中随机选择一个颜色
    color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
    
    scale0 = data["scale0"][0] if "scale0" in data else 1
    scale1 = data["scale1"][0] if "scale1" in data else 1
    
    # 调整坐标至两倍大小
    x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
    
    # 存储线段信息
    lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
    
    # 存储圆圈信息
    circles.append(((int(x0), int(y0)), color))
    circles.append(((int(x1) + W1, int(y1)), color))

# 先绘制所有线段
for start, end, color in lines:
    cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

# 再绘制所有圆圈
for center, color in circles:
    cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

# 保存图像到文件
output_path = 'sample_match.png'
cv2.imwrite(output_path, combined_image)

# 使用 matplotlib 显示拼接后的图像
plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
plt.axis('off')  # 关闭坐标轴
plt.show()

In [ ]:
import torch
import numpy as np

dump = np.load("dump/rcrm_baseline_outdoor/rcrm_pred_eval.npy", allow_pickle=True)

In [23]:
epi_errs = []
R_errs = []
t_errs = []
identifiers = []

for i in range(1500):
    epi_errs.append(dump[i]["epi_errs"])
    R_errs.append(dump[i]["R_errs"])
    t_errs.append(dump[i]["t_errs"])
    
R_errs = np.array(R_errs)
t_errs = np.array(t_errs)

In [ ]:
errors = np.max(np.stack([R_errs, t_errs]), axis=0)

In [26]:
errors = [0] + sorted(list([*errors]))
recall = list(np.linspace(0, 1, len(errors)))

aucs = []
thresholds = [5, 10, 20]

for thr in thresholds:
    last_index = np.searchsorted(errors, thr)
    y = recall[: last_index] + [recall[last_index - 1]]
    x = errors[: last_index] + [thr]
    aucs.append(np.trapz(y, x)/ thr)

In [ ]:
aucs